In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/imtkaggleteam/netflix/NetFlix.csv')

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# DAP --> Data Analysis Process

#data wrangling 

# 1. Data Gathering 
# 2. Data Assessing 
# 3. Data Preprocessing 

# Exploratory Analysis 

In [ ]:
df.head()

remove show_id as it is a high cardinality value, will not be able to help in further analysis

In [ ]:
# drop 
df.drop(['show_id','description'],axis=1,inplace=True)

In [ ]:
df.columns

In [ ]:
# overview 

df.head()

In [ ]:
# seeking info 

df.info()

### Netflix Dataset Overview

* The dataset contains **7,787 rows and 10 columns** after preprocessing.
* Initially, there were **12 columns**; **`show_id`** and **`description`** were removed due to low analytical relevance.
* It includes **2 numerical features** and **8 categorical (object-type) features**.
* The dataset contains **missing values**, which require appropriate handling through imputation or removal.
* The high proportion of categorical variables indicates the need for **encoding techniques** during further analysis or modeling.

### One-line Summary

The dataset is moderately sized, predominantly categorical, and requires preprocessing to handle missing values and prepare it for analysis.


In [ ]:
# seeking description

df.describe()

### Additional Insights

* The **release year** ranges from **1925 to 2021**, indicating that Netflix hosts content spanning several decades.

* However, the absence of records beyond 2021 suggests **data inconsistency or outdated data collection**, as the current year is 2026.

* The **duration column** ranges from **1 to 312**, which reflects mixed data types:

  * **Movies** are represented in minutes
  * **TV Shows** are represented in number of seasons

* This indicates that the duration feature is **not standardized** and may require **separation or transformation** for accurate analysis.


In [ ]:
# completeness 

df.isnull().sum()[df.isnull().sum()>0]

In [ ]:
df.isnull().mean()[df.isnull().mean()>0]*100

In [ ]:
# placeholders  --> unknown

df.director.mode()

df.director.fillna('unknown',inplace=True)

In [ ]:
df.isnull().mean()[df.isnull().mean()>0]*100

In [ ]:
import warnings
warnings.filterwarnings('ignore')
df.cast.fillna('unknown',inplace=True)
df.isnull().mean()[df.isnull().mean()>0]*100

In [ ]:
print(df.country.mode().values)
df.country.fillna('United States',inplace=True)
df.isnull().mean()[df.isnull().mean()>0]*100

In [ ]:
# dropna 

df.dropna(inplace=True)
print('percentage of null values ',df.isnull().mean()[df.isnull().mean()>0]*100)

In [ ]:
# exploratory analysis 

df.columns

# univariate analysis 
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px 
temp = df.type.value_counts().reset_index()
colors = ['#8C0A12', '#1B3A6F']
px.pie(temp,values='count',names='type',hole=.5,height=400,
       title='Distribution of type of content on Netflix',color_discrete_sequence=colors).show()
temp

* Approximately **70% of the content consists of movies**, while the remaining **30% comprises TV series**.

* This distribution clearly indicates that **Netflix’s content library is heavily dominated by movies**.

* It suggests that the platform primarily focuses on **movie-based content consumption**, with comparatively fewer TV shows available.

* The imbalance may reflect **user demand trends**, where movies are more frequently consumed or easier to distribute globally.

* From a data perspective, this also highlights a **content skew**, which should be considered during analysis or modeling to avoid biased insights.


In [ ]:
df.columns

df.release_year.describe()
# df.date_added

plt.figure(figsize=(12,4))
sns.histplot(data=df,x='release_year',kde=True,color= '#8C0A12')
plt.show()
plt.figure(figsize=(12,4))
sns.kdeplot(data=df,x='release_year',color= '#1B3A6F')
plt.show()

print('skewness ',df.release_year.skew())

* The **distribution of release years is negatively skewed (left-skewed)**, indicating that a large proportion of movies were released in more recent years.

* This suggests that **most of the content in the dataset has been produced after the year 2000**, reflecting the expansion of the film industry and digital distribution.

* The growth can be attributed to factors such as **advancements in technology, increased accessibility to storage media (e.g., DVDs and CDs), and the rise of global streaming platforms**.

* A noticeable **decline in movie releases is observed after 2020**, which can be linked to the impact of the **COVID-19 pandemic**, where production and theatrical releases were significantly disrupted.

* Overall, the trend highlights how **external factors like technological evolution and global events influence content production patterns**.


In [ ]:
temp_date = df.date_added.str.split('-',expand=True)[[2]].value_counts().sort_index().reset_index()

temp_date.columns = ['year','count']
temp_date
plt.figure(figsize=(12,4))
sns.lineplot(data=temp_date,x='year',y='count',color='#8C0A12')
plt.title('Number of content added as per year')
plt.grid()
plt.show()
print('-'*114)
plt.figure(figsize=(12,4))
sns.lineplot(data=temp_date.tail(8),x='year',y='count',color='#8C0A12')
plt.title('Number of content added as per year')
plt.grid()
plt.show()

* Netflix began its journey as an online streaming platform in **2007**, marking the start of digital content distribution.

* From **2008 to 2014**, the volume of content added each year remained relatively low, indicating a **slow expansion phase** of the platform.

* After **2014**, there is a significant and consistent increase in content additions, reflecting Netflix’s **aggressive growth strategy**.

* This surge can be attributed to factors such as **reduced internet costs, wider global internet accessibility, and increased adoption of streaming services**.

* The platform appears to reach its **peak content addition around 2019–2020**, highlighting a period of maximum expansion.

* A noticeable **decline is observed in 2020 and 2021**, which can be linked to the **COVID-19 pandemic**, as production and release schedules were disrupted globally.

* Overall, the trend demonstrates how **technological advancements and external global events directly influence content availability on streaming platforms**.


In [ ]:
temp_country = df.country.value_counts().head(15).reset_index()

colors = [
    '#67001F','#8C0A12','#B2182B','#D6604D','#F4A582',
    '#FDDBC7','#F7F7F7','#D1E5F0','#92C5DE','#4393C3',
    '#2166AC','#1B3A6F','#053061','#031F3D','#021124'
]

plt.figure(figsize=(12,5))

sns.countplot(
    data=df,
    y='country',
    order=temp_country['country'],
    palette=colors   # ✅ correct
)
plt.title('Top 15 Content-Producing Countries on Netflix')
plt.show()

* The **majority of content originates from the United States**, indicating that Netflix’s library is heavily influenced by Hollywood and US-based productions.

* The **second-largest contribution comes from India**, which is notable given that Netflix entered the Indian market **approximately 9 years after its initial launch in the US**. Despite the late entry, India has rapidly become a significant content source.

* Following India, there is a **sharp drop in content contribution from other countries**, such as the United Kingdom, Japan, and South Korea.

* Although **Japan and South Korea are globally recognized for anime and drama content**, their representation in the dataset is comparatively low.

* This suggests a **potential underrepresentation of regional content** or a selective acquisition strategy by Netflix, despite strong global demand for content from these countries.

* Overall, the distribution highlights a **strong geographical imbalance**, with the US dominating, emerging markets like India growing rapidly, and other content-rich regions contributing relatively less.


In [ ]:
df.columns
df.director.value_counts()
df[df.director=='Raúl Campos, Jan Suter']
df.director.value_counts()

In [ ]:
df.columns
print('max number of actors in a movie ',df.cast.str.split(',',expand=True).shape[1])

temp_cast = df.cast.str.split(',',expand=True)[[0]].value_counts().head(11).reset_index().iloc[1:11]
colors = ['#8C0A12','#B2182B','#D6604D','#F4A582','#FDD5C4','#C7E0ED','#92C5DE','#4393C3','#1B3A6F','#031F3D']
plt.figure(figsize=(12,4))
sns.barplot(data=temp_cast,y=0,x='count',palette=colors)
plt.title('Top 10 most frequent actors on Netflix')
plt.xticks(rotation=90)
plt.show()

* Some movies in the dataset have **large ensemble casts, with up to 50 actors**, indicating the presence of multi-starrer or highly collaborative productions.

* In terms of actor frequency, **Shah Rukh Khan** has the highest number of appearances, followed by **Akshay Kumar**, making them the most prominent actors in the dataset.

* **Adam Sandler** ranks third, highlighting Netflix’s strong association with his films, while **David Attenborough** appears fourth, primarily due to documentary content.

* A significant number of actors in the top list are from India, including **Amitabh Bachchan**, **Ajay Devgn**, and **Aamir Khan**.

* This trend indicates that **Indian actors dominate the frequency distribution**, suggesting a strong presence of Indian content on Netflix.

* Overall, the pattern reflects that **India is a key and growing market for Netflix**, both in terms of content production and audience demand.


In [ ]:
df.columns

temp_rating = df.rating.value_counts().head(10).reset_index()
colors = ['#8C0A12','#B2182B','#D6604D','#F4A582','#FDD5C4','#C7E0ED','#92C5DE','#4393C3','#1B3A6F','#031F3D']
plt.figure(figsize=(12,4))
sns.barplot(data=temp_rating,x='count',y='rating',palette=colors)
plt.title('top 10 most frequent ratings on Netflix')
plt.show()

* The **TV-MA rating has the highest number of listings**, indicating that mature content is the most prominently available and likely the most consumed on Netflix.

* This suggests that **adult audiences form a major segment of the platform’s user base**, driving demand for mature and diverse storytelling.

* The **TV-14 category ranks second**, highlighting that **teenagers and young adults are also highly active users** on Netflix.

* The presence of **TV-PG content in the third position** reflects that **family-friendly and child-appropriate content is also significant**, though comparatively lower than mature content.

* Overall, the rating distribution indicates that Netflix primarily caters to **teenagers and adults**, while still maintaining a **balanced offering for younger audiences**.


In [ ]:
df.columns

df['duration']

df.type

movie = df.query('type=="Movie"')
show1 = df.query('type=="TV Show"')

In [ ]:
l1 = ['#8C0A12', '#1B3A6F']

plt.figure(figsize=(12,3))
sns.histplot(data=movie,x='duration',color=l1[0])
plt.title('ditribution of duration(Movie)')
plt.show()
plt.figure(figsize=(12,3))
sns.boxplot(data=movie,x='duration',color=l1[1])
plt.title('ditribution of duration(Movie)')
plt.show()

print('-'*114)

plt.figure(figsize=(12,3))
sns.countplot(data=show1,x='duration',color=l1[0])
plt.title('ditribution of duration(TV Series)')
plt.show()
plt.figure(figsize=(12,3))
sns.boxplot(data=show1,x='duration',color=l1[1])
plt.title('ditribution of duration(TV Series)')
plt.show()

* The **distribution of movie durations is slightly right-skewed**, indicating that most movies have shorter to moderate runtimes, with fewer movies having very long durations.

* The **majority of movies fall within the range of 80 to 120 minutes**, suggesting this is the standard and most preferred duration for films on Netflix.

* The boxplot shows the presence of **outliers on both ends**, with some movies having extremely short durations and others exceeding **200–300 minutes**, representing special or extended content.

* The **median duration is close to ~100 minutes**, reinforcing that most content is centered around a typical feature-length format.

* For **TV series**, the distribution is highly **right-skewed**, with the majority having **only 1–2 seasons**, indicating that short-series formats are more common.

* As the number of seasons increases, the count drops significantly, showing that **long-running series (8+ seasons) are relatively rare**.

* The boxplot for TV series highlights **many outliers on the higher end**, meaning a few shows have significantly more seasons compared to the majority.

* Overall, Netflix content suggests a preference for **moderate-length movies and shorter TV series**, aligning with modern audience attention spans and binge-watching behavior.


In [ ]:
df.columns
temp_genre = df.genres.str.split(',',expand=True)[[0]].value_counts().sort_values(ascending=False).head(10).reset_index()
temp_genre.columns = ['genre','count']

colors = ['#8C0A12','#B2182B','#D6604D','#F4A582','#FDD5C4','#C7E0ED','#92C5DE','#4393C3','#1B3A6F','#031F3D']
plt.figure(figsize=(12,4))
sns.barplot(data=temp_genre,x='genre',y='count',palette=colors)
plt.title('top 10 most frequent genre on Netflix')
plt.xticks(rotation=45)
plt.show()

temp_genre

* **Dramas dominate**, showing strong preference for story-based content.
* **Comedies are second**, indicating high demand for light entertainment.
* **Documentaries & Action** have solid presence, reflecting diverse viewer interests.
* **International TV shows are high**, highlighting Netflix’s global expansion.
* **Kids & family content is moderate**, not the primary focus.
* **Crime & stand-up comedy** serve niche but consistent audiences.
* **Horror is least**, showing limited but specific demand.


## 📊 Summary

This Exploratory Data Analysis (EDA) on the Netflix dataset provides insights into content distribution, production trends, and audience preferences.

- The dataset contains **7,700+ titles**, including both movies and TV shows, with **movies dominating the platform**.
- A majority of the content has been released **post-2000**, indicating Netflix’s focus on modern productions.
- The **United States leads in content production**, followed by **India**, which shows significant growth.
- There was a **rapid increase in content addition after 2014**, aligning with Netflix’s global expansion.
- The platform mainly targets **mature audiences**, with **TV-MA being the most frequent rating**.
- **Movies** generally have a duration of **80–120 minutes**.
- **TV Shows** are mostly **short (1–2 seasons)**, indicating a preference for limited series.
- Popular genres and actors reflect a mix of **global and Indian entertainment trends**.

---

## 💡 Recommendations

Based on the analysis, the following recommendations are suggested:

- **Expand regional content** to include more countries beyond the US and India.
- Invest more in **short-format content** like mini-series and limited series.
- **Leverage popular actors and genres** to increase viewer engagement.
- Maintain a better **balance in content ratings** by including more family-friendly content.
- **Clean and standardize dataset features** (like duration) for improved analysis and modeling.
- Continue focusing on **emerging markets**, especially India, to sustain growth.

---

## 🧾 Conclusion

The analysis reveals that Netflix’s content strategy is strongly driven by **modern trends, audience demand, and global expansion**.

- The platform heavily focuses on **recent, movie-based, and mature content**.
- The **US dominates content creation**, while **India is rapidly emerging** as a key contributor.
- The surge in content after 2014 highlights Netflix’s evolution into a **global streaming giant**.

However, there is still room for improvement in terms of:
- **Content diversity**
- **Regional representation**
- **Data consistency**